In [0]:
# Databricks notebook source
# SILVER LAYER: Fact Results Table
# Purpose: Join all 4 bronze tables into one enriched, trusted fact table

from pyspark.sql.functions import (
    col, when, lit, coalesce, current_timestamp,
    trim, upper, round as spark_round,
    regexp_replace, cast, sum as spark_sum
)
from pyspark.sql.types import DoubleType, IntegerType

# ============ 1. READ FROM BRONZE ============
results_raw  = spark.read.table("f1_data.bronze.results")
drivers_dim  = spark.read.table("f1_data.silver.dim_drivers")   # Use cleaned Silver dims
sessions_dim = spark.read.table("f1_data.silver.dim_sessions")
meetings_dim = spark.read.table("f1_data.silver.dim_meetings")

print(f"Bronze results: {results_raw.count()}")

# ============ 2. CLEAN RESULTS ============

# gap_to_leader is STRING (e.g. "+5.432", "+1 LAP", "0.000")
# Need to handle multiple formats

results_clean = results_raw \
    .withColumn("driver_number", col("driver_number").cast(IntegerType())) \
    .withColumn("points", col("points").cast(DoubleType())) \
    .withColumn("position", col("position").cast(DoubleType())) \
    .withColumn("duration", col("duration").cast(DoubleType())) \
    .withColumn("number_of_laps", col("number_of_laps").cast(IntegerType())) \
    .withColumn("dnf", coalesce(col("dnf"), lit(False))) \
    .withColumn("dns", coalesce(col("dns"), lit(False))) \
    .withColumn("dsq", coalesce(col("dsq"), lit(False))) \
    .withColumn("points", coalesce(col("points"), lit(0.0))) \
    .withColumn("gap_to_leader_raw", col("gap_to_leader")) \
    .withColumn("gap_to_leader_seconds",
        # Extract numeric gap only ("+5.432" → 5.432); laps/DNF → null
        when(
            col("gap_to_leader").rlike("^[+]?[0-9]+\\.[0-9]+$"),
            regexp_replace(col("gap_to_leader"), "^[+]", "").cast(DoubleType())
        ).otherwise(lit(None).cast(DoubleType()))
    ) \
    .withColumn("gap_type",
        # Categorize the gap: normal, lapped, or leader
        when(col("gap_to_leader").rlike("LAP"), lit("LAPPED"))
        .when(col("gap_to_leader") == "0.000", lit("LEADER"))
        .when(col("gap_to_leader").rlike("^[+]?[0-9]"), lit("TIME_GAP"))
        .when(col("dnf") == True, lit("DNF"))
        .when(col("dns") == True, lit("DNS"))
        .when(col("dsq") == True, lit("DSQ"))
        .otherwise(lit("UNKNOWN"))
    )

# ============ 3. VALIDATE RESULTS ============
# Critical: driver_number, meeting_key, session_key must exist for joins
results_valid = results_clean.filter(
    col("driver_number").isNotNull() &
    col("meeting_key").isNotNull() &
    col("session_key").isNotNull()
)

results_rejected = results_clean.filter(
    col("driver_number").isNull() |
    col("meeting_key").isNull() |
    col("session_key").isNull()
)

if results_rejected.count() > 0:
    print(f"⚠️  {results_rejected.count()} results rejected (missing join keys)")
    results_rejected \
        .withColumn("rejection_reason", lit("missing_join_keys")) \
        .write.mode("append").format("delta").saveAsTable("f1_data.silver._rejected_records")

# ============ 4. JOIN ALL TABLES ============

# Step 1: Results + Drivers (core join)
results_with_drivers = results_valid.alias("r").join(
    drivers_dim.alias("d"),
    on=(
        (col("r.driver_number") == col("d.driver_number")) &
        (col("r.meeting_key")   == col("d.meeting_key"))   &
        (col("r.session_key")   == col("d.session_key"))
    ),
    how="left"  # Left join: keep result even if driver not found
)

# Step 2: Add session context
results_with_sessions = results_with_drivers.join(
    sessions_dim.alias("s"),
    on=(
        (col("r.session_key") == col("s.session_key")) &
        (col("r.meeting_key") == col("s.meeting_key"))
    ),
    how="left"
)

# Step 3: Add meeting/circuit context
results_full = results_with_sessions.join(
    meetings_dim.alias("m"),
    on=(col("r.meeting_key") == col("m.meeting_key")),
    how="left"
)

# ============ 5. DERIVE ENRICHMENT COLUMNS ============
results_enriched = results_full \
    # A boolean value that states whether the driver finished the race or not
    .withColumn("finished_race",
        when(
            (col("r.dnf") == False) & (col("r.dns") == False) & (col("r.dsq") == False),
            lit(True)
        ).otherwise(lit(False))
    ) \
    
    # A boolean value that states whether the driver made it to the podium
    .withColumn("on_podium",
        when(col("r.position").isNotNull() & (col("r.position") <= 3), lit(True))
        .otherwise(lit(False))
    ) \

    # A column that states the average points per lap, taken by the number of points over the number laps
    .withColumn("points_per_lap",
        when(
            (col("r.number_of_laps").isNotNull()) & (col("r.number_of_laps") > 0),
            spark_round(col("r.points") / col("r.number_of_laps"), 4)
        ).otherwise(lit(None).cast(DoubleType()))
    ) \

    # A boolean value that states if the driver has a full name (?)
    .withColumn("driver_found_in_dim",
        when(col("d.full_name").isNotNull(), lit(True)).otherwise(lit(False))
    )

# ============ 6. FLAG JOIN QUALITY ============
# How complete are your joins? (This is a key Silver-layer metric)
# Convert the boolean column driver_found_in_dim into integers
# Adds up all the 1s to get the total count of matched drivers and names the resulting column drivers_matched
# Runs this sum aggregation over your entire DataFrame
# Pulls the aggregated data from the cluster memory to your local machine as a list, and selects the first row
join_quality = results_enriched.agg(
    spark_sum(col("driver_found_in_dim").cast("int")).alias("drivers_matched"),
).collect()[0]

total = results_enriched.count()
print(f"\n📊 JOIN QUALITY REPORT")
print(f"  Total results: {total}")
print(f"  Drivers matched: {results_enriched.filter(col('driver_found_in_dim') == True).count()}")
print(f"  Drivers unmatched: {results_enriched.filter(col('driver_found_in_dim') == False).count()}")

# ============ 7. FINAL COLUMN SELECTION ============
fact_results_silver = results_enriched.select(
    # Keys
    col("r.driver_number"),
    col("r.meeting_key"),
    col("r.session_key"),

    # Driver context (from dim_drivers)
    col("d.full_name").alias("driver_name"),
    col("d.first_name").alias("driver_first_name"),
    col("d.last_name").alias("driver_last_name"),
    col("d.name_acronym").alias("driver_acronym"),
    col("d.team_name"),
    col("d.team_colour"),
    col("d.country_code").alias("driver_country_code"),

    # Session context (from dim_sessions)
    col("s.session_name"),
    col("s.session_type"),
    col("s.date_start").alias("session_date"),
    col("s.year"),

    # Circuit context (from dim_meetings)
    col("m.meeting_name"),
    col("m.meeting_official_name"),
    col("m.circuit_short_name"),
    col("m.circuit_type"),
    col("m.country_name").alias("race_country"),
    col("m.location").alias("race_location"),

    # Raw result data
    col("r.position"),
    col("r.points"),
    col("r.duration"),
    col("r.number_of_laps"),
    col("r.dnf"),
    col("r.dns"),
    col("r.dsq"),
    col("r.gap_to_leader_raw"),
    col("r.gap_to_leader_seconds"),
    col("r.gap_type"),

    # Derived columns
    col("finished_race"),
    col("on_podium"),
    col("points_per_lap"),
    col("driver_found_in_dim"),

    # Metadata
    lit("f1_data.bronze.results").alias("silver_source"),
    current_timestamp().alias("silver_processed_at"),
    lit("PASSED").alias("data_quality_flag")
)

# ============ 8. WRITE TO SILVER ============
fact_results_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", "true") \
    .partitionBy("year", "session_type") \
    .saveAsTable("f1_data.silver.fact_results")

print(f"\n✓ Silver fact_results written: {fact_results_silver.count()} records")
print(f"✓ Partitioned by: year, session_type")

Bronze results: 419

📊 JOIN QUALITY REPORT
  Total results: 419
  Drivers matched: 419
  Drivers unmatched: 0

✓ Silver fact_results written: 419 records
✓ Partitioned by: year, session_type
